# How close is it?

Every example digitised as the help's walkthroughs do it - with the same marks and options - and compared with the
values the figure really shows (`dwb.known`, the workbench's comparison; `compare-known` in the Terminal). This takes
a minute or so: Figure 2 has 59 rows of labels to read.

In [ ]:
import os, time, warnings
import pandas as pd
import trace_digitiser as td
from dwb import known

EX = "/data/examples/hearn-whitmore-2026"
RECIPES = [
    ("figure-1a", "lines", dict(trace_colours=["#4682b4"], exclude=[(120, 15, 262, 58)])),
    ("figure-1a", "lines", dict(trace_colours=["#ff8c00"], exclude=[(990, 15, 1296, 58)], y_ref=[(485.5, 6), (34.5, 13)])),
    ("figure-1a", "lines", dict(trace_colours=["#228b22"], plot_area=(543, 20, 876, 159),
                                x_ref=[(558, 1972), (865, 1979)], y_ref=[(62, 60000), (118, 50000)])),
    ("figure-1b", "bars", dict()),
    ("figure-2", "rows", dict(row_values=list(range(1955, 2014)), bins=24, dots="black", row_spacing_value=1.1)),
    ("figure-3a", "lines", dict()),
    ("figure-3b", "polar", dict(colours=["#1f77b4"], r_ref=[(5, 1955), (250, 2005)])),
    ("figure-3c", "points", dict()),
    ("figure-4", "heatmap", dict(rows=21)),
    ("figure-s3", "points", dict()),
]
READER = {"lines": td.digitise, "points": td.digitise_points, "bars": td.digitise_bars, "rows": td.digitise_rows,
          "heatmap": td.digitise_heatmap, "polar": td.digitise_polar}

In [ ]:
rows = []
for figure, kind, options in RECIPES:
    t = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = READER[kind](f"{EX}/{figure}.png", write_outputs=False, **options)
    data = result.trace_data if kind == "lines" else result.data
    c = known.compare(figure, data, kind)
    for label, s in c["stats"].items():
        if s.get("n"):
            rows.append({"figure": figure, "kind": kind, "what": label, "compared": s["n"],
                         "typical error": round(s["median_abs"], 5), "% of the axis": round(s["median_abs_pct"], 2),
                         "seconds": round(time.time() - t, 1)})
summary = pd.DataFrame(rows)
summary

*Typical error* is the median of the absolute differences, in the units of the axis, and *% of the axis* the
same as a percentage of the axis's span. Most are a small fraction of a pixel. The exceptions say something about
the figures:

- **Figure 1a, births**: the line is dense - each column of pixels holds more than a month - so the middle of what is
  drawn in a column is not any one month's value.
- **Figure 2**: each row's fill is a few pixels high, so a pixel is a large part of it.
- **Figure 3b, angles**: only the points found are compared; where markers overlap, some years are missed and other
  points are found. The walkthrough shows how to put that right by hand (*Edit points*).
- **Figure 4**: each column of pixels is nearly two of the heat map's columns, which are averaged here.